# 🚀 Notebook 9: Training & Text Generation

**Bringing Your LLM to Life**

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. Implement a **complete training loop**
2. Understand **optimization** (AdamW)
3. Monitor **training metrics**
4. Generate text with **different sampling strategies**
5. Understand **temperature**, **top-k**, and **top-p**
6. Train and evaluate a **working LLM**

---

## 📚 Table of Contents

1. [Setup & Data Loading](#1-setup-data-loading)
2. [Training Loop](#2-training-loop)
3. [Loss Monitoring](#3-loss-monitoring)
4. [Text Generation Strategies](#4-text-generation-strategies)
5. [Training the Model](#5-training-the-model)
6. [Generating Text](#6-generating-text)
7. [Final Thoughts](#7-final-thoughts)

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pickle
import os
import time

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seed
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

### Environment Setup (Colab/Local)

This cell detects whether you're running in Google Colab or locally and sets up the environment accordingly.

In [ ]:
# ========================================
# ENVIRONMENT DETECTION & SETUP
# ========================================

# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

# Setup directories
if IN_COLAB:
    # Create necessary directories for Colab
    os.makedirs('data', exist_ok=True)
    os.makedirs('visualizations/tokenization', exist_ok=True)
    os.makedirs('visualizations/embeddings', exist_ok=True)
    os.makedirs('visualizations/data_pipeline', exist_ok=True)
    print("Created directories")
    
    # Set paths for Colab
    DATA_DIR = 'data'
    VIZ_DIR = 'visualizations'
else:
    # Use relative paths for local execution
    DATA_DIR = '../data'
    VIZ_DIR = '../visualizations'
    
    # Create directories if they don't exist (LOCAL FIX)
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(f'{VIZ_DIR}/tokenization', exist_ok=True)
    os.makedirs(f'{VIZ_DIR}/embeddings', exist_ok=True)
    os.makedirs(f'{VIZ_DIR}/data_pipeline', exist_ok=True)
    print(f"Created directories: {DATA_DIR}, {VIZ_DIR}")

print(f"\nData directory: {DATA_DIR}")
print(f"Visualization directory: {VIZ_DIR}")

---

## 1. Setup & Data Loading

### 📊 Load Configuration and Data

In [ ]:
# Load configuration
with open(f'{DATA_DIR}/data_config.pkl', 'rb') as f:
    config = pickle.load(f)

vocab_size = config['vocab_size']
block_size = config['block_size']
batch_size = config['batch_size']

# Load data
train_data = torch.load(f'{DATA_DIR}/train_data.pt')
val_data = torch.load(f'{DATA_DIR}/val_data.pt')

# Load embedding config
checkpoint = torch.load(f'{DATA_DIR}/embedding_layer.pth')
n_embd = checkpoint['n_embd']

print(f"📊 Configuration:")
print(f"Vocabulary size: {vocab_size}")
print(f"Block size:      {block_size}")
print(f"Batch size:      {batch_size}")
print(f"Embedding dim:   {n_embd}")
print(f"\n📊 Data:")
print(f"Train data size: {len(train_data):,} tokens")
print(f"Val data size:   {len(val_data):,} tokens")

### 🏗️ Load Model Architecture

In [ ]:
# Import model components from previous notebook

class SelfAttentionHead(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        self.head_size = head_size
    
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) / (self.head_size ** 0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, num_heads, block_size, dropout=0.1):
        super().__init__()
        assert n_embd % num_heads == 0
        self.heads = nn.ModuleList([
            SelfAttentionHead(n_embd, n_embd // num_heads, block_size, dropout)
            for _ in range(num_heads)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd, expansion_factor=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd * expansion_factor),
            nn.GELU(),
            nn.Linear(n_embd * expansion_factor, n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, num_heads, block_size, dropout)
        self.ffn = FeedForward(n_embd, expansion_factor=4, dropout=dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, n_layer, num_heads, dropout=0.1):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[
            TransformerBlock(n_embd, num_heads, block_size, dropout)
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.block_size = block_size
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        
        return idx

print("✅ Model architecture loaded!")

### 🎯 Create Model Instance

In [ ]:
# Model hyperparameters
n_layer = 4
num_heads = 4
dropout = 0.2

# Create model
model = GPTLanguageModel(
    vocab_size=vocab_size,
    n_embd=n_embd,
    block_size=block_size,
    n_layer=n_layer,
    num_heads=num_heads,
    dropout=dropout
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())

print(f"🎉 Model created and moved to {device}!")
print(f"\n📊 Model Configuration:")
print(f"   Vocabulary size: {vocab_size}")
print(f"   Embedding dim:   {n_embd}")
print(f"   Block size:      {block_size}")
print(f"   Num layers:      {n_layer}")
print(f"   Num heads:       {num_heads}")
print(f"   Dropout:         {dropout}")
print(f"\n💾 Total parameters: {total_params:,}")

---

## 2. Training Loop

### 🎯 Data Batch Function

In [ ]:
def get_batch(split):
    """
    Generate a batch of data.
    
    Args:
        split: 'train' or 'val'
    
    Returns:
        x: Input batch, shape (batch_size, block_size)
        y: Target batch, shape (batch_size, block_size)
    """
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# Test batch generation
xb, yb = get_batch('train')
print(f"📊 Batch shapes:")
print(f"   Input (x):  {xb.shape}")
print(f"   Target (y): {yb.shape}")
print(f"\n✅ Batch generation working!")

### ⚙️ Optimizer Setup

In [ ]:
# Training hyperparameters
learning_rate = 3e-4
max_iters = 5000
eval_interval = 500
eval_iters = 200

# Create optimizer (AdamW)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f"⚙️ Optimizer: AdamW")
print(f"📊 Training Configuration:")
print(f"   Learning rate:   {learning_rate}")
print(f"   Max iterations:  {max_iters:,}")
print(f"   Eval interval:   {eval_interval}")
print(f"   Eval iterations: {eval_iters}")

---

## 3. Loss Monitoring

### 📊 Estimate Loss Function

In [ ]:
@torch.no_grad()
def estimate_loss():
    """
    Estimate loss on train and val sets.
    
    Returns:
        out: Dictionary with 'train' and 'val' losses
    """
    out = {}
    model.eval()
    
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    
    model.train()
    return out

# Test loss estimation
print("📊 Testing loss estimation...")
losses = estimate_loss()
print(f"\nInitial losses:")
print(f"   Train: {losses['train']:.4f}")
print(f"   Val:   {losses['val']:.4f}")
print(f"\n💡 Random model should have loss ≈ -ln(1/{vocab_size}) = {-np.log(1/vocab_size):.4f}")

---

## 4. Text Generation Strategies

### 🎯 Temperature Scaling

**Temperature** controls randomness:
- `T = 1.0`: Normal sampling
- `T < 1.0`: More confident (sharper distribution)
- `T > 1.0`: More random (flatter distribution)

```
logits_scaled = logits / temperature
probs = softmax(logits_scaled)
```

In [ ]:
# Visualize temperature effect
logits = torch.tensor([2.0, 1.0, 0.5, 0.1])
temperatures = [0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, temp in zip(axes, temperatures):
    scaled_logits = logits / temp
    probs = F.softmax(scaled_logits, dim=0).numpy()
    
    ax.bar(range(len(probs)), probs, alpha=0.7, edgecolor='black', linewidth=2)
    ax.set_xlabel('Token', fontsize=11, fontweight='bold')
    ax.set_ylabel('Probability', fontsize=11, fontweight='bold')
    ax.set_title(f'Temperature = {temp}', fontsize=13, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/embeddings/temperature_effect.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 Lower temperature = more confident, Higher temperature = more random")

### 🎯 Top-k Sampling

Only sample from the **k most likely tokens**:

```
1. Get top-k logits
2. Set all other logits to -∞
3. Apply softmax
4. Sample from top-k
```

---

## 5. Training the Model

### 🚀 Main Training Loop

In [ ]:
# Training loop
train_losses = []
val_losses = []
iterations = []

print("🚀 Starting training...\n")
start_time = time.time()

for iter in range(max_iters):
    
    # Evaluate loss periodically
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        iterations.append(iter)
        
        elapsed = time.time() - start_time
        print(f"Step {iter:5d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f} | Time: {elapsed:.1f}s")
    
    # Sample a batch
    xb, yb = get_batch('train')
    
    # Forward pass
    logits, loss = model(xb, yb)
    
    # Backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

total_time = time.time() - start_time
print(f"\n✅ Training complete!")
print(f"Total time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final val loss:   {val_losses[-1]:.4f}")

### 📊 Plot Training Curves

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(iterations, train_losses, 'o-', linewidth=2, markersize=6, label='Train Loss', color='steelblue')
ax.plot(iterations, val_losses, 's-', linewidth=2, markersize=6, label='Val Loss', color='coral')

ax.set_xlabel('Iteration', fontsize=12, fontweight='bold')
ax.set_ylabel('Loss', fontsize=12, fontweight='bold')
ax.set_title('Training Progress', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/embeddings/training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 Loss should decrease over time!")
print(f"Loss reduction: {train_losses[0]:.4f} → {train_losses[-1]:.4f}")

---

## 6. Generating Text

### 🎯 Load Character Mappings

In [ ]:
# Load character mappings
with open(f'{DATA_DIR}/char_mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)

stoi = mappings['stoi']
itos = mappings['itos']

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

print("✅ Character mappings loaded!")
print(f"\nVocabulary: {list(itos.values())[:20]}...")

### 🎨 Generate with Different Temperatures

In [ ]:
# Generate text with different temperatures
model.eval()

prompt = "The "
context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

temperatures = [0.5, 1.0, 1.5]

print("🎨 Generating text with different temperatures:\n")
print(f"Prompt: '{prompt}'\n")
print("=" * 80)

for temp in temperatures:
    generated = model.generate(context, max_new_tokens=200, temperature=temp)
    text = decode(generated[0].tolist())
    
    print(f"\nTemperature = {temp}:")
    print("-" * 80)
    print(text)
    print("-" * 80)

### 🎯 Generate with Top-k Sampling

In [ ]:
# Generate with top-k sampling
print("\n🎯 Generating text with top-k sampling:\n")
print(f"Prompt: '{prompt}'\n")
print("=" * 80)

top_k_values = [5, 10, 20]

for k in top_k_values:
    context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    generated = model.generate(context, max_new_tokens=200, temperature=1.0, top_k=k)
    text = decode(generated[0].tolist())
    
    print(f"\nTop-k = {k}:")
    print("-" * 80)
    print(text)
    print("-" * 80)

### 💾 Save Trained Model

In [ ]:
# Save the trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'iterations': iterations,
    'config': {
        'vocab_size': vocab_size,
        'n_embd': n_embd,
        'block_size': block_size,
        'n_layer': n_layer,
        'num_heads': num_heads,
        'dropout': dropout
    }
}, f'{DATA_DIR}/trained_model.pth')

print("✅ Trained model saved to f'{DATA_DIR}/trained_model.pth'")

---

## 7. Final Thoughts

### 🎉 Congratulations!

You've successfully built a **Large Language Model from scratch**!

### ✅ What You've Accomplished:

1. **Tokenization**
   - Character-level tokenizer
   - Byte Pair Encoding (BPE)
   - Vocabulary building

2. **Data Pipeline**
   - Batching mechanism
   - Train/val split
   - Efficient data loading

3. **Embeddings**
   - Token embeddings
   - Positional embeddings
   - Combined representations

4. **Attention Mechanism**
   - Self-attention
   - Multi-head attention
   - Causal masking

5. **Feed-Forward Networks**
   - Position-wise transformations
   - GELU activation
   - Expansion and projection

6. **Transformer Architecture**
   - Residual connections
   - Layer normalization
   - Complete transformer blocks

7. **Training & Generation**
   - Training loop
   - Loss monitoring
   - Text generation strategies

### 🚀 Next Steps:

1. **Experiment with hyperparameters**:
   - Increase model size (more layers, larger embedding)
   - Try different learning rates
   - Adjust dropout

2. **Train on larger datasets**:
   - Use more text data
   - Train for more iterations
   - Implement learning rate scheduling

3. **Advanced techniques**:
   - Implement top-p (nucleus) sampling
   - Add gradient clipping
   - Try different optimizers

4. **Scale up**:
   - Use GPU acceleration
   - Implement mixed precision training
   - Distribute training across multiple GPUs

### 📚 Key Takeaways:

- **Transformers are powerful**: Self-attention enables long-range dependencies
- **Architecture matters**: Residuals + LayerNorm enable deep networks
- **Training is iterative**: Monitor losses, adjust hyperparameters
- **Generation is creative**: Temperature and sampling strategies control output

### 🎓 You Now Understand:

- How GPT models work internally
- Why certain architectural choices are made
- How to implement each component from scratch
- How to train and generate text

### 🌟 Final Words:

You've built a real, working LLM! While it's smaller than GPT-3 (175B parameters), the **core principles are identical**. The same architecture, scaled up with more data and compute, powers the most advanced language models today.

**Keep experimenting, keep learning, and keep building!** 🚀

In [ ]:
# Final summary
print("="*80)
print("🎉 CONGRATULATIONS! 🎉")
print("="*80)
print("\nYou've successfully built a Large Language Model from scratch!")
print("\n📊 Final Statistics:")
print(f"   Model parameters:  {total_params:,}")
print(f"   Training time:     {total_time/60:.1f} minutes")
print(f"   Final train loss:  {train_losses[-1]:.4f}")
print(f"   Final val loss:    {val_losses[-1]:.4f}")
print(f"   Loss reduction:    {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.1f}%")
print("\n🚀 Keep building and experimenting!")
print("="*80)